# 03 — InceptionV3 Standalone Model

InceptionV3 transfer learning model trained on 299×299 MRI images for 4-class brain tumor classification.
Two-stage fine-tuning: Stage 1 trains the classification head with the base frozen; Stage 2 unfreezes
the last 20 InceptionV3 layers for domain adaptation.
Saves `inceptionv3_model.h5` to `../saved_models/` for use in 04_InceptionV3_Ensemble.ipynb and 08_AllModelsCombined.ipynb.

## Section 0: Google Colab Setup & Dataset Download

Detects Colab, installs kaggle/huggingface_hub, downloads the Brain Tumor MRI dataset once (shared across all notebooks), and sets path variables.

**Colab Secrets required** (Runtime → Manage secrets):
- `KAGGLE_USERNAME` and `KAGGLE_KEY` — from kaggle.com/settings/account
- `HF_TOKEN` — from huggingface.co/settings/tokens

In [ ]:
import sys, os, json

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Google Colab detected — setting up environment...")

    # Install extra dependencies
    os.system("pip install kaggle huggingface_hub -q")

    # Kaggle credentials from Colab Secrets
    try:
        from google.colab import userdata
        _kaggle_user = userdata.get('KAGGLE_USERNAME')
        _kaggle_key  = userdata.get('KAGGLE_KEY')
        HF_TOKEN     = userdata.get('HF_TOKEN')
    except Exception:
        # Fallback — replace these values if Colab Secrets are not configured
        _kaggle_user = 'YOUR_KAGGLE_USERNAME'   # ← replace
        _kaggle_key  = 'YOUR_KAGGLE_KEY'         # ← replace
        HF_TOKEN     = 'YOUR_HF_TOKEN'           # ← replace

    # Write kaggle.json
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as _f:
        json.dump({'username': _kaggle_user, 'key': _kaggle_key}, _f)
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    # Download dataset once — all notebooks share the same /content/MRI_DATASET/
    DATASET_PATH     = "/content/MRI_DATASET/"
    SAVED_MODELS_DIR = "/content/saved_models/"
    RESULTS_DIR      = "/content/results/"

    if not os.path.exists(DATASET_PATH + "Training"):
        print("Downloading Brain Tumor MRI dataset from Kaggle (≈ 150 MB)...")
        os.system(f"kaggle datasets download masoudnickparvar/brain-tumor-mri-dataset -p /content/")
        os.system(f"unzip -q /content/brain-tumor-mri-dataset.zip -d {DATASET_PATH}")
        os.system("rm -f /content/brain-tumor-mri-dataset.zip")
        print("✓ Dataset downloaded and extracted")
    else:
        print("✓ Dataset already present — skipping download")

else:
    print("Running locally")
    DATASET_PATH     = "../MRI_DATASET/"
    SAVED_MODELS_DIR = "../saved_models/"
    RESULTS_DIR      = "../results/"
    HF_TOKEN         = os.environ.get('HF_TOKEN', '')

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"  Dataset path : {DATASET_PATH}")
print(f"  Models path  : {SAVED_MODELS_DIR}")
print(f"  Results path : {RESULTS_DIR}")
print("✓ Environment ready")

## Section 1: Imports & Configuration

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
print("✓ Imports complete")

## Section 2: Constants & Hyperparameters

In [ ]:
NOTEBOOK_NAME = "03_InceptionV3_Standalone"

# Dataset
DATASET_PATH     = globals().get("DATASET_PATH", "../MRI_DATASET/")
TRAIN_DIR        = DATASET_PATH + "Training/"
TEST_DIR         = DATASET_PATH + "Testing/"

# Classes — FIXED ORDER, DO NOT CHANGE
CLASS_NAMES      = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES      = 4

# InceptionV3 requires 299×299 input
IMG_HEIGHT       = 299
IMG_WIDTH        = 299
CHANNELS         = 3

# Training hyperparameters
BATCH_SIZE       = 32
EPOCHS_STAGE1    = 10   # Frozen base: train top layers only
EPOCHS_STAGE2    = 10   # Unfrozen: fine-tune last 20 layers
LEARNING_RATE    = 1e-4
FINE_TUNE_LR     = 1e-5
VALIDATION_SPLIT = 0.2

# Reproducibility
RANDOM_SEED      = 42

# Paths
SAVED_MODELS_DIR = globals().get("SAVED_MODELS_DIR", "../saved_models/")
RESULTS_DIR      = globals().get("RESULTS_DIR", "../results/")
RESULTS_NB_DIR   = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
HF_TOKEN         = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_NB_DIR, exist_ok=True)

# Set seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print("✓ Constants configured")
print(f"  Image size : {IMG_HEIGHT}×{IMG_WIDTH} (InceptionV3 requires 299×299)")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Stage 1 epochs: {EPOCHS_STAGE1} (frozen base)")
print(f"  Stage 2 epochs: {EPOCHS_STAGE2} (fine-tune last 20 layers)")
print(f"  Classes    : {CLASS_NAMES}")

## Section 3: Data Loading & Verification

In [ ]:
# --- Augmentation for training data ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='training',
    seed=RANDOM_SEED,
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='validation',
    seed=RANDOM_SEED,
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print("=" * 50)
print("DATA VERIFICATION")
print(f"Class indices      : {train_generator.class_indices}")
print(f"Training samples   : {train_generator.samples}")
print(f"Validation samples : {val_generator.samples}")
print(f"Test samples       : {test_generator.samples}")
print(f"Image size         : {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"Batch size         : {BATCH_SIZE}")
print("=" * 50)

## Section 4: Data Preprocessing & Augmentation

In [ ]:
# Augmentation is defined in the ImageDataGenerator in Section 3.
print("✓ Data augmentation configured via ImageDataGenerator")

## Section 5: Model Definition

In [ ]:
# --- Load InceptionV3 base with ImageNet weights ---
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS))
base_model.trainable = False   # Freeze all base layers during Stage 1
print(f"✓ InceptionV3 base loaded. Total layers: {len(base_model.layers)}")

# --- Add classification head ---
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu', name='feature_layer')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer')(x)

model = models.Model(inputs=base_model.input, outputs=output, name='inceptionv3_model')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"✓ Model compiled for Stage 1 (lr={LEARNING_RATE})")
model.summary()

## Section 6: Model Training

In [ ]:
# ─── STAGE 1: Train classification head (base frozen) ───────────────────────────────────────
print("=" * 50)
print("STAGE 1: Training top layers (base frozen)")
print("=" * 50)

callbacks_stage1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath=SAVED_MODELS_DIR + 'inceptionv3_stage1_best.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

history1 = model.fit(
    train_generator,
    epochs=EPOCHS_STAGE1,
    validation_data=val_generator,
    callbacks=callbacks_stage1,
    verbose=1
)
print("✓ Stage 1 complete")

# ─── STAGE 2: Fine-tune last 20 layers ────────────────────────────────────────────
print("\n" + "=" * 50)
print("STAGE 2: Fine-tuning last 20 layers of InceptionV3 base")
print("=" * 50)

# Unfreeze last 20 layers of the base
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f"✓ Recompiled for Stage 2 (lr={FINE_TUNE_LR})")

callbacks_stage2 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath=SAVED_MODELS_DIR + 'inceptionv3_stage2_best.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1)
]

history2 = model.fit(
    train_generator,
    epochs=EPOCHS_STAGE2,
    validation_data=val_generator,
    callbacks=callbacks_stage2,
    verbose=1
)
print("✓ Stage 2 (fine-tuning) complete")

# --- Combine histories and plot ---
combined_acc  = history1.history['accuracy']  + history2.history['accuracy']
combined_val  = history1.history['val_accuracy'] + history2.history['val_accuracy']
combined_loss = history1.history['loss'] + history2.history['loss']
combined_vloss= history1.history['val_loss'] + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(combined_acc,  label='Train Accuracy')
ax1.plot(combined_val,  label='Val Accuracy')
ax1.axvline(x=EPOCHS_STAGE1, color='r', linestyle='--', label='Fine-tune start')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(combined_loss,  label='Train Loss')
ax2.plot(combined_vloss, label='Val Loss')
ax2.axvline(x=EPOCHS_STAGE1, color='r', linestyle='--', label='Fine-tune start')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.suptitle(f'{NOTEBOOK_NAME} — Training History (Stage 1 + Stage 2)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_NB_DIR, f"{NOTEBOOK_NAME}_training_history.jpg"),
            dpi=150, bbox_inches='tight')
plt.show()

## Section 7: Model Evaluation

In [ ]:
import re as _re

def evaluate_model(model, generator, model_name="Model"):
    """Standard evaluation: confusion matrix, classification report, ROC, PR curves."""
    _fname = _re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    generator.reset()
    y_pred_proba = model.predict(generator, verbose=1)
    y_pred       = np.argmax(y_pred_proba, axis=1)
    y_true       = generator.classes

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} — Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_confusion_matrix.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    # Classification Report
    print(f"\n{model_name} — Classification Report")
    print("=" * 60)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    # ROC Curve
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} — ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_roc_curve.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    # Precision-Recall Curve
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_proba[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_pred_proba[:, i])
        plt.plot(recall, precision, label=f'{cls} (AP = {ap:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} — Precision-Recall Curve')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_pr_curve.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    return y_pred, y_pred_proba

# Evaluate on test set
y_pred, y_pred_proba = evaluate_model(model, test_generator, model_name=NOTEBOOK_NAME)
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"\n✓ Test Accuracy : {test_acc:.4f}")
print(f"✓ Test Loss     : {test_loss:.4f}")

## Section 8: Save Model

In [ ]:
model_path = SAVED_MODELS_DIR + 'inceptionv3_model.h5'
model.save(model_path)
print(f"✓ Model saved to: {model_path}")
print("  This model is used as:")
print("  1. Standalone InceptionV3 classifier (08_AllModelsCombined.ipynb)")
print("  2. Feature extractor backbone (04_InceptionV3_Ensemble.ipynb)")

## Section 9: Results Summary

In [ ]:
print("=" * 60)
print(f"NOTEBOOK: {NOTEBOOK_NAME}")
print(f"Dataset  : {train_generator.samples + val_generator.samples} training images")
print(f"Classes  : {CLASS_NAMES}")
print(f"Image size: {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Stage 1: {EPOCHS_STAGE1} epochs (lr={LEARNING_RATE}, base frozen)")
print(f"Stage 2: {EPOCHS_STAGE2} epochs (lr={FINE_TUNE_LR}, last 20 layers unfrozen)")
print(f"Seed     : {RANDOM_SEED}")
print("-" * 60)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")
print("=" * 60)
print("Saved models location:", SAVED_MODELS_DIR)

## Section 10: HuggingFace Upload

Uploads the model file(s) saved in Section 8 and all result JPGs from Section 7 to `shehank98/brain-tumor-mri-models` on HuggingFace Hub.

Requires `HF_TOKEN` to be set (via Colab Secrets in Section 0, or `HF_TOKEN` env var).

In [ ]:
# ── HuggingFace repository ────────────────────────────────────────────────────
HF_REPO_ID = "shehank98/brain-tumor-mri-models"   # your HF repo

try:
    from huggingface_hub import HfApi, login as hf_login
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'])
    from huggingface_hub import HfApi, login as hf_login

if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set — skipping HuggingFace upload.")
    print("  Set it in Colab Secrets (key: HF_TOKEN) or as an env var.")
else:
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi()

    # Create repo if it does not exist yet
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model",
                    private=False, exist_ok=True)
    print(f"✓ Repository ready: https://huggingface.co/{HF_REPO_ID}")

    # Upload model files
    _model_files = ['inceptionv3_model.h5']
    for _fname in _model_files:
        _local = os.path.join(SAVED_MODELS_DIR, _fname)
        if not os.path.exists(_local):
            print(f"  SKIP (not found): {_fname}")
            continue
        _size = os.path.getsize(_local) / 1e6
        print(f"  Uploading {_fname} ({_size:.1f} MB)...", end="", flush=True)
        api.upload_file(
            path_or_fileobj=_local,
            path_in_repo=f"models/{_fname}",
            repo_id=HF_REPO_ID,
            commit_message=f"Upload {_fname} from 03_InceptionV3",
        )
        print(" done")

    # Upload results JPGs
    _results_nb = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
    if os.path.exists(_results_nb):
        _jpgs = [f for f in os.listdir(_results_nb) if f.endswith('.jpg')]
        for _jpg in sorted(_jpgs):
            print(f"  Uploading result chart {_jpg}...", end="", flush=True)
            api.upload_file(
                path_or_fileobj=os.path.join(_results_nb, _jpg),
                path_in_repo=f"results/{NOTEBOOK_NAME}/{_jpg}",
                repo_id=HF_REPO_ID,
                commit_message=f"Add result chart {_jpg}",
            )
            print(" done")
        print(f"✓ {len(_jpgs)} result charts uploaded")
    else:
        print("  No result charts found — run Section 7 first")

    print(f"\n✓ Upload complete: https://huggingface.co/{HF_REPO_ID}")